# Single Distribution Center Solution

Note formulates the transportation problem of the O'Reilly Auto Parts network with the addition of features to enforce the concept of each store being satisfied by a single distribution center.

In [2]:
import pandas as pd
import pyomo.environ as pe

import model_utils as utils

The below cell generates the model shown below.  Note the the following changes with respect to the usual transportation problem model.

- It is now necessary to introduce binary decision variables to capture when a distribution center is used to satisfy a store. These are shown below with the $\delta_{d,s}$ variables.

- The third constraint in the model limits the number of distribution centers for each store to be 1.  Note the right hand side could be modified to relax this constraint for specific stores or to set a higher maximum value.

- The fourth and fifth constraints act as links between the binary decision variables and the real decision variables which represent the units shipped between stores and distribution centers.  Note the use of the Big M method where M is taken to be 10,000.
    - The fourth constraint achieves the logical condition $$\sum_{p \in P} x_{d,s,p} \gt 0 \implies \delta = 1$$
    - The fifth constraint achieves the logical condition $$\delta = 1 \implies \sum_{p \in P} x_{d,s,p} \ge 0.001$$

**Model**

*Sets*  
Let D be the set of distribution centers.  $D = \{d_1, d_2, d_3, ...\}$  
Let S be the set of retail stores.  $S = \{s_1, s_2, s_3, ...\}$  
Let P be the set of products.  $P = \{p_1, p_2, p_3, ...\}$  

*Params*

Amount of inventory of product p at distribution center d

$$ a_{d,p} \in \mathbb{Z} \quad \forall d \in D, \forall p \in P $$

Demand of product p at store s

$$ b_{s,p} \in \mathbb{Z} \quad \forall s \in S, \forall p \in P $$

Unit cost of shipping product p from distribution center d to store s

$$ c_{d,s,p} \in \mathbb{R} \quad \forall d \in D, \forall s \in S, \forall p \in P $$

*Variables*  

Quantity of product p to ship from distribution center d to store s  

$$x_{d,s,p} \in \mathbb{R} \hspace{2mm} \forall d \in D, \forall s \in S, \forall p \in P$$

Decision to ship products from distribution store d to store s

$$ \delta_{d,s} \in \{0,1\} \hspace{2mm} \forall d \in D, \forall s \in S  $$

*Model*

$$
\begin{array}{cccccl} \text{min} & \sum_{d \in D} \sum_{s \in S} \sum_{p \in P} x_{d,s,p}c_{d,s,p} & & & \\\\
 \text{s.t.} & & & & \\\\
 & \sum_{d \in D} x_{d,s,p} & = & b_{s,p} & \forall s \in S, \forall p \in P & \text{Satisfy store demand} \\\\
 & \sum_{s \in S} x_{d,s,p} & \le & a_{d,p} & \forall d \in D, \forall p \in P & \text{Inventory limit at distribution center} \\\\
 & \sum_{d \in D} \delta_{d,s} & \le & 1 & \forall s \in S & \text{Limit of 1 distribution center per store} \\\\
 & \sum_{p \in P} x_{d,s,p} & \le & M\delta_{d,s} & \forall d \in D, \forall s \in S & \text {Indicator Linking Constraint 1} \\\\
 & \sum_{p \in P} x_{d,s,p} & \ge & 0.001\delta_{d,s} & \forall d \in D, \forall s \in S & \text {Indicator Linking Constraint 2} \\\\
 & x_{d,s,p} & \ge & 0 & \forall d \in D, \forall s \in S, \forall p \in P
\end{array}
$$

In [3]:
model = utils.gen_model(True)

The below cell solves the model from above.  Note that this solution requires more computational effort and optimal solution takes nearly 10 minutes to be determined.

In [4]:
solver = pe.SolverFactory('cbc')
results = solver.solve(model, tee=True)

Welcome to the CBC MILP Solver 
Version: 2.10.7 
Build Date: Feb 14 2022 

command line - /usr/bin/cbc -printingOptions all -import /tmp/tmp2d_6zp8n.pyomo.lp -stat=1 -solve -solu /tmp/tmp2d_6zp8n.pyomo.soln (default strategy 1)
Option for printingOptions changed from normal to all
Presolve 3601 (-119) rows, 6666 (-714) columns and 25434 (-2856) elements
Statistics for presolved model
Original problem has 1230 integers (1230 of which binary)
Presolved problem has 1230 integers (1230 of which binary)
==== 1230 zero objective 5437 different
==== absolute objective values 5437 different
==== for integers 1230 zero objective 1 different
1230 variables have objective of 0
==== for integers absolute objective values 1 different
1230 variables have objective of 0
===== end objective counts


Problem has 3601 rows, 6666 columns (5436 with objective) and 25434 elements
Column breakdown:
5436 of type 0.0->inf, 0 of type 0.0->up, 0 of type lo->inf, 
0 of type lo->up, 0 of type free, 0 of type fixe

## Solution Review

The below cells interrogate the optimal solution.  Note that the transportation costs have now risen to $7,481.53 with the requirement of restricting a store to a single distribution center.  The final cell shows a portion of the optimal solution.  Note how now store 1016 (Knoxville, TN) is sourcing products solely from a distribution center in Forest Park, GA.  This differs from the baseline solution where it sourced products from distribution centers in Greensboro, NC and Lebanon, TN.

In [5]:
print(f"Transportation Cost: ${pe.value(model.obj):,.2f}")

Transportation Cost: $7,481.53


In [9]:
plan_df = utils.annotate_model_plan(model)
plan_df.sort_values(by=['store'], inplace=True)
plan_df[:25]

,product,product_desc,dist_center,dist_center_loc,store,store_city_loc,qty
472,16024,Hopkins SubZero 21-1/2 Inch Snow Brush,7,"Forest Park, GA",1016,"Knoxville, TN",54.0
642,17228,Hopkins SubZero Auto Emergency Shovel,7,"Forest Park, GA",1016,"Knoxville, TN",39.0
795,20,Scotwood Industries Traction Melt Driveway De-...,7,"Forest Park, GA",1016,"Knoxville, TN",32.0
89,AS242,Prestone Windshield De-Icer,7,"Forest Park, GA",1016,"Knoxville, TN",59.0
284,AS240,Prestone 15.5 Ounce Windshield De-Icer,7,"Forest Park, GA",1016,"Knoxville, TN",62.0
784,20,Scotwood Industries Traction Melt Driveway De-...,7,"Forest Park, GA",1021,"Rome, GA",70.0
633,17228,Hopkins SubZero Auto Emergency Shovel,7,"Forest Park, GA",1021,"Rome, GA",22.0
274,AS240,Prestone 15.5 Ounce Windshield De-Icer,7,"Forest Park, GA",1021,"Rome, GA",83.0
78,AS242,Prestone Windshield De-Icer,7,"Forest Park, GA",1021,"Rome, GA",116.0
461,16024,Hopkins SubZero 21-1/2 Inch Snow Brush,7,"Forest Park, GA",1021,"Rome, GA",104.0
